# Train one fold

In [ ]:
from pathlib import Path

import torch

from knee.dataset import build_data
from knee.model import Model2p5D
from knee.train import TrainConfig, fit, pick_device, seed_all, write_history, write_oof

In [ ]:
cfg = TrainConfig().apply({
    "CACHE_KEY": "cache_v3",
    "CSV_DIR": "data",
    "OUT_DIR": "runs/local",
    "FOLD": 0,
    "SLOTS": [0, 1, 2],
    "NUM_WORKERS": 4,
})
out_dir = Path(cfg.OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
cfg

## Data


In [ ]:
seed_all(cfg.SEED)
device = pick_device()
train_dl, valid_dl = build_data(cfg)
len(train_dl.dataset), len(valid_dl.dataset), device

## Model


In [ ]:
model = Model2p5D(cfg.as_dict()).to(device)
sum(p.numel() for p in model.parameters()) / 1e6

## Fit

validate after every epoch


In [ ]:
result = fit(cfg, model, train_dl, valid_dl, out_dir)
oof = write_oof(cfg, result, valid_dl.dataset.df, out_dir)
write_history(cfg, result, out_dir)

In [ ]:
import pandas as pd

pd.DataFrame(result.history)